In [1]:
import pandas as pd
import numpy as np

In [2]:
ENTREPOT_PATH = '/home/tbadie/Bureau/data/data_entrepot_outils/'
donnees = {}

def import_dfs(df_names, path_data, sep = ','):
    i = 0
    for df_name in df_names: 
        donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, low_memory=False).replace({'\r\n': '\n'}, regex=True)

tables = [
    'sdc',
    'synthetise',
    'connection_synthetise',
    'noeuds_synthetise',
    'connection_realise',
    'noeuds_realise',
    'zone',
    'parcelle',
    'composant_culture',
    'espece',

    'entite_unique_par_sdc_nettoyage',

    'connection_synthetise_restructure',
    'noeuds_synthetise_restructure',

    'poids_connexions_synthetise_rotation',
    'poids_noeuds_realise',

    'typologie_culture_outils_dirodur',
    'date_de_semis_outils_dirodur'
    ]

# import des données
import_dfs(tables, ENTREPOT_PATH, sep = ',')

In [3]:
def initialisation_donnees(donnees):

    poids_S = donnees['poids_connexions_synthetise_rotation'][['connexion_id','poids_conx_agregation_norm_synth']].rename(columns={'connexion_id':'connection_synthetise_id'}).copy()
    poids_R = donnees['poids_noeuds_realise'][['noeuds_realise_id','poids_surface_developpee_normalisee']].copy()
    date_semis = donnees['date_de_semis_outils_dirodur'][['culture_id','saison_semis_detect_via_intv']].copy()

    sdc = donnees['sdc'][['id','filiere']].rename(columns={'id':'sdc_id'}).copy()

    unique_sdc = donnees['entite_unique_par_sdc_nettoyage'].copy()
    sdc_real = sdc.loc[sdc['sdc_id'].isin(unique_sdc.loc[unique_sdc['entite_retenue'] == 'realise_retenu','sdc_id'])]
    synthetise = donnees['synthetise'][['id','sdc_id']].rename(columns={'id':'synthetise_id'}).copy()
    synthetise = synthetise.loc[synthetise['synthetise_id'].isin(unique_sdc['entite_retenue'].unique())]

    cnx_s = donnees['connection_synthetise'][['id','cible_noeuds_synthetise_id']].rename(columns={'id':'connection_synthetise_id', 'cible_noeuds_synthetise_id':'noeuds_synthetise_id'}).copy()
    cnx_s_rest = donnees['connection_synthetise_restructure'].rename(columns={'id':'connection_synthetise_id'}).copy()
    nd_s = donnees['noeuds_synthetise'][['id','synthetise_id']].rename(columns={'id':'noeuds_synthetise_id'}).copy()
    nd_s_rest = donnees['noeuds_synthetise_restructure'].rename(columns={'id':'noeuds_synthetise_id'}).copy()

    cnx_r = donnees['connection_realise'][['id','cible_noeuds_realise_id','culture_intermediaire_id']].rename(columns={'id':'connexion_realise_id','cible_noeuds_realise_id':'noeuds_realise_id'})
    nd_r = donnees['noeuds_realise'].rename(columns={'id':'noeuds_realise_id'}).copy()
    zone = donnees['zone'][['id','parcelle_id']].rename(columns={'id':'zone_id'}).copy()
    parcelle = donnees['parcelle'][['id','sdc_id']].rename(columns={'id':'parcelle_id'}).copy()

    cropsp = donnees['composant_culture'][['id','espece_id','culture_id']].rename(columns={'id':'composant_culture_id'}).copy()
    sp = donnees['espece'][['id','libelle_espece_botanique','typodirodur_espece','typodirodur_espece_precise','typodirodur_espece_famille_bota','typodirodur_espece_periode_semis']].rename(columns={'id':'espece_id'}).copy()
    typo_dirodur = donnees['typologie_culture_outils_dirodur'][['culture_id', 'typodirodur_culture', 'culture_est_avec_compagne', 
                                                                'culture_est_annuelle_asso', 'culture_est_prairie', 'typo_cpg']].copy()
    sp = cropsp.merge(sp, how = 'left', on = 'espece_id')

    sp['ponderation_composant'] = 1/sp.groupby('culture_id')['composant_culture_id'].transform("count")

    # merge outer pour les noeud sur les connexion en réalisé car tous les noeuds n'ont pas forcément de connexion
    # merge inner avec synthetise et sdc pour n'avoir que les entite unique par sdc !
    itk_s = cnx_s.merge(cnx_s_rest, how='left', on='connection_synthetise_id').merge(nd_s, how='left', on='noeuds_synthetise_id').merge(nd_s_rest, how='left', on='noeuds_synthetise_id').merge(synthetise, how='inner', on='synthetise_id')
    itk_r = cnx_r.merge(nd_r, how='outer', on ='noeuds_realise_id').merge(zone, how='left', on='zone_id').merge(parcelle, how='left', on='parcelle_id').merge(sdc_real, how='inner', on='sdc_id')
    itk = pd.concat([itk_s, itk_r])

    itk = itk[['connection_synthetise_id', 'noeuds_realise_id', 'culture_id', 'culture_intermediaire_id', 'synthetise_id', 'sdc_id']]
    composant_itk = itk.merge(sp, on='culture_id', how='left').merge(typo_dirodur, how = 'left', on = 'culture_id')
    composant_itk = composant_itk.merge(poids_S, how='left', on='connection_synthetise_id')
    composant_itk = composant_itk.merge(poids_R, how='left', on='noeuds_realise_id')

    composant_itk = composant_itk.merge(date_semis, how='left', on='culture_id')
    composant_itk['saison_semis_detect_via_intv'] = composant_itk['typodirodur_espece_periode_semis'].fillna(composant_itk['saison_semis_detect_via_intv'])
    composant_itk.drop(columns = 'saison_semis_detect_via_intv', inplace=True)

    # On calcule les poids par composant au seins du sdc (ou synthetise). On prends le poids de connexion ou le poids de noeuds selon la méthode de saisie (R ou S)
    composant_itk['poids_composant_dans_sdc'] = np.where(composant_itk['connection_synthetise_id'].notna(),
                                                        composant_itk['ponderation_composant'] * composant_itk['poids_conx_agregation_norm_synth'],
                                                        composant_itk['ponderation_composant'] * composant_itk['poids_surface_developpee_normalisee'])

    return composant_itk

df = initialisation_donnees(donnees)

In [4]:
def richness(p):
    return len(p.index)

def shannon(p):
    sh = -(p * np.log2(p)).sum()
    if sh == -0:
        return 0
    return sh

def evenness(p):
    s = len(p)
    if s <= 1:
        return np.nan
    return shannon(p) / np.log2(s)

def simpson(p):
    return (p**2).sum()

# def gini_simpson(p):
#     return 1 - simpson(p)

def inverse_simpson(p):
    s = simpson(p)
    if pd.isna(s) or s == 0:
        return np.nan
    return 1 / s

def proportions(p, typology_col):
    return (
        p.pivot(index='sdc_id',
                columns=typology_col,
                values=p.index)
         .fillna(0)
         .add_prefix("prop_")
    )



def compute_typology_metrics(df, typology_col, prefix, cols_needed_for_proportion=None):
    # Il a certaines cultures en absentes (==> poids = NaN) comme souvent pour les Précédents fictifs par exemple
    df = df[df["poids_composant_dans_sdc"].notna()]

    # On ajoute la modalité Inconnu pour ne pas sous ou sur estimé les proportions des autres modalités (groupby excluant par défaut les NaN dasn la typology_col)
    df.loc[:,typology_col] = df[typology_col].fillna("Inconnu")
    proportions = df.groupby(typology_col)["poids_composant_dans_sdc"].sum()

    # Il y a potentiellement des modalité avec une somme de proportion à 0%, on les retire
    proportions =  proportions[proportions > 0]

    # Le sdc n'a pas les poids associés à chaque culture ou n'avait que des poids à 0% ou que des Nan
    if proportions.empty and typology_col == 'typodirodur_espece' :       
        return pd.Series({
            f"{prefix}_richesse": int(0),
            f"{prefix}_shannon": np.nan,
            f"{prefix}_evenness": np.nan,
            f"{prefix}_simpson": np.nan,
            f"{prefix}_inverse_simpson": np.nan,
        })
    elif proportions.empty and typology_col != 'typodirodur_espece' :       
        return pd.Series({
            f"{prefix}_richesse": int(0),
            f"{prefix}_shannon": np.nan,
        })
    
    # Calculs des indicateurs
    proportions = proportions / proportions.sum()

    if typology_col == 'typodirodur_espece' :
        metrics = {
            f"{prefix}_richesse": int(richness(proportions)),
            f"{prefix}_shannon": shannon(proportions),
            f"{prefix}_evenness": evenness(proportions),
            f"{prefix}_simpson": simpson(proportions),
            f"{prefix}_inverse_simpson": inverse_simpson(proportions),
            f"{prefix}_proportion_max": max(proportions),
        }
    else : 
        metrics = {
            f"{prefix}_richesse": int(richness(proportions)),
            f"{prefix}_shannon": shannon(proportions),
        }

    # Calculs des proportions
    # Cas des famille botanique, on combine la proportion de toutes les autres familles qu les 3 principales
    if typology_col == 'typodirodur_espece_famille_bota' :
        mask = proportions.index.isin(["Poaceae", "Fabaceae", "Brassicaceae"])
        others = proportions[~mask].sum()
        proportions = proportions[mask].copy()
        proportions["Autres"] = others

    if cols_needed_for_proportion is not None:
        for category, proportion in proportions.items():
            if category in cols_needed_for_proportion:
                metrics[f"{prefix}_{category}"] = proportion

    return metrics

In [5]:
result = (
    df.groupby("sdc_id")
    .apply(
        lambda sdc: pd.DataFrame([
            {
                **compute_typology_metrics(sdc, "typodirodur_espece", "typo_espece"),
                **compute_typology_metrics(sdc, "libelle_espece_botanique", "espece_bota"),
                **compute_typology_metrics(sdc, "typodirodur_espece_famille_bota", "famille_bota", ["Poaceae", "Fabaceae", "Brassicaceae", 'Autres']),
                **compute_typology_metrics(sdc, "typodirodur_espece_periode_semis", "saison_semis", ["printemps", "ete", "automne", 'hiver', 'pluriannuel']),
                "prop_culture_avec_compagne": sdc.loc[sdc["culture_est_avec_compagne"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_association": sdc.loc[sdc["culture_est_annuelle_asso"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_prairie": sdc.loc[sdc["culture_est_prairie"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_culture_intermédiaire": sdc.loc[sdc["culture_intermediaire_id"].notna(), "poids_composant_dans_sdc"].sum(),
                "prop_culture_porte_graine": sdc.loc[sdc["typo_cpg"].notna(), "poids_composant_dans_sdc"].sum(),
            }
        ]),
        include_groups=False,
    )
    .reset_index()
).drop(columns='level_1')


for col in [col for col in result.columns if 'richesse' in col.lower()]:
    result[col] = result[col].astype('Int64')
    
result

,sdc_id,typo_espece_richesse,typo_espece_shannon,typo_espece_evenness,typo_espece_simpson,typo_espece_inverse_simpson,typo_espece_proportion_max,espece_bota_richesse,espece_bota_shannon,famille_bota_richesse,...,saison_semis_ete,saison_semis_hiver,saison_semis_printemps,prop_culture_avec_compagne,prop_association,prop_prairie,prop_culture_intermédiaire,prop_culture_porte_graine,famille_bota_Brassicaceae,saison_semis_automne
0,fr.inra.agrosyst.api.entities.GrowingSystem_00...,6,2.449552,0.947616,0.192501,5.194768,0.248072,7,2.616162,2,...,0.200514,0.174807,0.128535,0.0,0.0,0.496144,0.128535,0.0,NaN,NaN
1,fr.inra.agrosyst.api.entities.GrowingSystem_00...,2,1.000000,1.000000,0.500000,2.000000,0.500000,2,1.000000,2,...,NaN,NaN,1.000000,0.0,0.0,0.000000,1.000000,0.0,NaN,NaN
2,fr.inra.agrosyst.api.entities.GrowingSystem_00...,2,1.000000,1.000000,0.500000,2.000000,0.500000,2,1.000000,1,...,0.500000,0.500000,NaN,0.0,0.0,0.000000,0.250000,0.0,NaN,NaN
3,fr.inra.agrosyst.api.entities.GrowingSystem_00...,11,3.263737,0.943432,0.113500,8.810573,0.150000,11,3.263737,3,...,0.100000,0.400000,0.050000,0.0,0.2,0.450000,0.000000,0.0,NaN,NaN
4,fr.inra.agrosyst.api.entities.GrowingSystem_00...,3,1.475336,0.930833,0.383450,2.607902,0.500000,3,1.475336,1,...,0.185000,0.500000,0.315000,0.0,0.0,0.000000,0.500000,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22212,fr.inra.agrosyst.api.entities.GrowingSystem_ff...,5,2.281073,0.982405,0.212375,4.708651,0.297001,5,2.281073,3,...,0.372864,0.468066,NaN,0.0,0.0,0.000000,0.000000,0.0,0.159070,0.159070
22213,fr.inra.agrosyst.api.entities.GrowingSystem_ff...,9,2.790504,0.880306,0.176484,5.666224,0.275000,11,3.213862,2,...,0.125000,0.375000,NaN,0.0,0.0,0.375000,0.000000,0.0,NaN,NaN
22214,fr.inra.agrosyst.api.entities.GrowingSystem_ff...,7,2.351843,0.837744,0.235425,4.247629,0.367946,7,2.351843,3,...,0.243843,0.446990,0.053451,0.0,0.0,0.108874,0.051353,0.0,0.146843,0.146843
22215,fr.inra.agrosyst.api.entities.GrowingSystem_ff...,2,0.918296,0.918296,0.555556,1.800000,0.666667,2,0.918296,2,...,0.333333,NaN,NaN,0.0,0.0,0.000000,0.000000,0.0,NaN,NaN


In [6]:
list_sdc_id = list(result.sample(50)['sdc_id'])

In [7]:
TEST_PATH = '/home/tbadie/Bureau/catalogue_script_agrosyst/02_outils/tests/data/test_get_indicateur_diversite_outils_dirodur/'

# Créer les nouvelles données filtrées
sdc2 = donnees['sdc'].loc[donnees['sdc']['id'].isin(list_sdc_id)]

synthetise2 = donnees['synthetise'].loc[donnees['synthetise']['sdc_id'].isin(sdc2['id'])]
ndS2 = donnees['noeuds_synthetise'].loc[donnees['noeuds_synthetise']['synthetise_id'].isin(synthetise2['id'])]
cxS2 = donnees['connection_synthetise'].loc[(donnees['connection_synthetise']['cible_noeuds_synthetise_id'].isin(ndS2['id'])) |  
                                            (donnees['connection_synthetise']['source_noeuds_synthetise_id'].isin(ndS2['id']))   ]

parcelle2 = donnees['parcelle'].loc[donnees['parcelle']['sdc_id'].isin(sdc2['id'])]
zone2 = donnees['zone'].loc[donnees['zone']['parcelle_id'].isin(parcelle2['id'])]
ndR2 = donnees['noeuds_realise'].loc[donnees['noeuds_realise']['zone_id'].isin(zone2['id'])]
cxR2 = donnees['connection_realise'].loc[(donnees['connection_realise']['cible_noeuds_realise_id'].isin(ndR2['id'])) |  
                                         (donnees['connection_realise']['source_noeuds_realise_id'].isin(ndR2['id']))   ]

ent_unique = donnees['entite_unique_par_sdc_nettoyage'].loc[donnees['entite_unique_par_sdc_nettoyage']['sdc_id'].isin(sdc2['id'])]
cxS2_rest = donnees['connection_synthetise_restructure'].loc[donnees['connection_synthetise_restructure']['id'].isin(cxS2['id'])]
ndS2_rest = donnees['noeuds_synthetise_restructure'].loc[donnees['noeuds_synthetise_restructure']['id'].isin(ndS2['id'])]
poids_cxS2 = donnees['poids_connexions_synthetise_rotation'].loc[donnees['poids_connexions_synthetise_rotation']['connexion_id'].isin(cxS2['id'])]
poids_ndR2 = donnees['poids_noeuds_realise'].loc[donnees['poids_noeuds_realise']['noeuds_realise_id'].isin(ndR2['id'])]

culture_id2 = set(ndS2_rest['culture_id'].tolist() + ndR2['culture_id'].tolist())

typo_diro = donnees['typologie_culture_outils_dirodur'].loc[donnees['typologie_culture_outils_dirodur']['culture_id'].isin(culture_id2)]
date_semis = donnees['date_de_semis_outils_dirodur'].loc[donnees['date_de_semis_outils_dirodur']['culture_id'].isin(culture_id2)]
cc2 = donnees['composant_culture'].loc[donnees['composant_culture']['culture_id'].isin(culture_id2)]
esp2 = donnees['espece'].loc[donnees['espece']['id'].isin(cc2['espece_id'])]


# imprimer les nouvelles données filtrées
sdc2.to_csv(TEST_PATH + 'sdc.csv', index=False, sep=',')

synthetise2.to_csv(TEST_PATH + 'synthetise.csv', index=False, sep=',')
ndS2.to_csv(TEST_PATH + 'noeuds_synthetise.csv', index=False, sep=',')
cxS2.to_csv(TEST_PATH + 'connection_synthetise.csv', index=False, sep=',')

parcelle2.to_csv(TEST_PATH + 'parcelle.csv', index=False, sep=',')
zone2.to_csv(TEST_PATH + 'zone.csv', index=False, sep=',')
ndR2.to_csv(TEST_PATH + 'noeuds_realise.csv', index=False, sep=',')
cxR2.to_csv(TEST_PATH + 'connection_realise.csv', index=False, sep=',')

ent_unique.to_csv(TEST_PATH + 'entite_unique_par_sdc_nettoyage.csv', index=False, sep=',')
cxS2_rest.to_csv(TEST_PATH + 'connection_synthetise_restructure.csv', index=False, sep=',')
ndS2_rest.to_csv(TEST_PATH + 'noeuds_synthetise_restructure.csv', index=False, sep=',')
poids_cxS2.to_csv(TEST_PATH + 'poids_connexions_synthetise_rotation.csv', index=False, sep=',')
poids_ndR2.to_csv(TEST_PATH + 'poids_noeuds_realise.csv', index=False, sep=',')

typo_diro.to_csv(TEST_PATH + 'typologie_culture_outils_dirodur.csv', index=False, sep=',')
date_semis.to_csv(TEST_PATH + 'date_de_semis_outils_dirodur.csv', index=False, sep=',')
cc2.to_csv(TEST_PATH + 'composant_culture.csv', index=False, sep=',')
esp2.to_csv(TEST_PATH + 'espece.csv', index=False, sep=',')

In [8]:
del donnees
donnees = {}
import_dfs(tables, TEST_PATH, sep = ',')

In [9]:
df = initialisation_donnees(donnees)

def richness(p):
    return len(p.index)

def shannon(p):
    sh = -(p * np.log2(p)).sum()
    if sh == -0:
        return 0
    return sh

def evenness(p):
    s = len(p)
    if s <= 1:
        return np.nan
    return shannon(p) / np.log2(s)

def simpson(p):
    return (p**2).sum()

# def gini_simpson(p):
#     return 1 - simpson(p)

def inverse_simpson(p):
    s = simpson(p)
    if pd.isna(s) or s == 0:
        return np.nan
    return 1 / s

def proportions(p, typology_col):
    return (
        p.pivot(index='sdc_id',
                columns=typology_col,
                values=p.index)
         .fillna(0)
         .add_prefix("prop_")
    )


def compute_typology_metrics(df, typology_col, prefix, cols_needed_for_proportion=None):
    # Il a certaines cultures en absentes (==> poids = NaN) comme souvent pour les Précédents fictifs par exemple
    df = df[df["poids_composant_dans_sdc"].notna()]

    # On ajoute la modalité Inconnu pour ne pas sous ou sur estimé les proportions des autres modalités (groupby excluant par défaut les NaN dasn la typology_col)
    df.loc[:,typology_col] = df[typology_col].fillna("Inconnu")
    proportions = df.groupby(typology_col)["poids_composant_dans_sdc"].sum()

    # Il y a potentiellement des modalité avec une somme de proportion à 0%, on les retire
    proportions =  proportions[proportions > 0]

    # Le sdc n'a pas les poids associés à chaque culture ou n'avait que des poids à 0% ou que des Nan
    if proportions.empty and typology_col == 'typodirodur_espece' :       
        return pd.Series({
            f"{prefix}_richesse": int(0),
            f"{prefix}_shannon": np.nan,
            f"{prefix}_evenness": np.nan,
            f"{prefix}_simpson": np.nan,
            f"{prefix}_inverse_simpson": np.nan,
        })
    elif proportions.empty and typology_col != 'typodirodur_espece' :       
        return pd.Series({
            f"{prefix}_richesse": int(0),
            f"{prefix}_shannon": np.nan,
        })
    
    # Calculs des indicateurs
    proportions = proportions / proportions.sum()

    if typology_col == 'typodirodur_espece' :
        metrics = {
            f"{prefix}_richesse": int(richness(proportions)),
            f"{prefix}_shannon": shannon(proportions),
            f"{prefix}_evenness": evenness(proportions),
            f"{prefix}_simpson": simpson(proportions),
            f"{prefix}_inverse_simpson": inverse_simpson(proportions),
            f"{prefix}_proportion_max": max(proportions),
        }
    else : 
        metrics = {
            f"{prefix}_richesse": int(richness(proportions)),
            f"{prefix}_shannon": shannon(proportions),
        }

    # Calculs des proportions
    # Cas des famille botanique, on combine la proportion de toutes les autres familles qu les 3 principales
    if typology_col == 'typodirodur_espece_famille_bota' :
        mask = proportions.index.isin(["Poaceae", "Fabaceae", "Brassicaceae"])
        others = proportions[~mask].sum()
        proportions = proportions[mask].copy()
        proportions["Autres"] = others

    if cols_needed_for_proportion is not None:
        for category, proportion in proportions.items():
            if category in cols_needed_for_proportion:
                metrics[f"{prefix}_{category}"] = proportion

    return metrics

In [10]:
result = (
    df.groupby("sdc_id")
    .apply(
        lambda sdc: pd.DataFrame([
            {
                **compute_typology_metrics(sdc, "typodirodur_espece", "typo_espece"),
                **compute_typology_metrics(sdc, "libelle_espece_botanique", "espece_bota"),
                **compute_typology_metrics(sdc, "typodirodur_espece_famille_bota", "famille_bota", ["Poaceae", "Fabaceae", "Brassicaceae", 'Autres']),
                **compute_typology_metrics(sdc, "typodirodur_espece_periode_semis", "saison_semis", ["printemps", "ete", "automne", 'hiver', 'pluriannuel']),
                "prop_culture_avec_compagne": sdc.loc[sdc["culture_est_avec_compagne"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_association": sdc.loc[sdc["culture_est_annuelle_asso"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_prairie": sdc.loc[sdc["culture_est_prairie"] == "oui", "poids_composant_dans_sdc"].sum(),
                "prop_culture_intermédiaire": sdc.loc[sdc["culture_intermediaire_id"].notna(), "poids_composant_dans_sdc"].sum(),
                "prop_culture_porte_graine": sdc.loc[sdc["typo_cpg"].notna(), "poids_composant_dans_sdc"].sum(),
            }
        ]),
        include_groups=False,
    )
    .reset_index()
).drop(columns='level_1')

for col in [col for col in result.columns if 'richesse' in col.lower()]:
    result[col] = result[col].astype('Int64')

result

,sdc_id,typo_espece_richesse,typo_espece_shannon,typo_espece_evenness,typo_espece_simpson,typo_espece_inverse_simpson,typo_espece_proportion_max,espece_bota_richesse,espece_bota_shannon,famille_bota_richesse,...,saison_semis_hiver,prop_culture_avec_compagne,prop_association,prop_prairie,prop_culture_intermédiaire,prop_culture_porte_graine,saison_semis_printemps,famille_bota_Brassicaceae,famille_bota_Fabaceae,saison_semis_automne
0,fr.inra.agrosyst.api.entities.GrowingSystem_04...,5,2.000000,0.861353,0.312500,3.200000,0.500000,5,2.000000,1,...,0.500000,0.000,0.000000,0.000000,0.000000,0.00,NaN,NaN,NaN,NaN
1,fr.inra.agrosyst.api.entities.GrowingSystem_06...,1,0.000000,NaN,1.000000,1.000000,1.000000,1,0.000000,1,...,NaN,0.000,0.000000,0.000000,0.000000,0.00,NaN,NaN,NaN,NaN
2,fr.inra.agrosyst.api.entities.GrowingSystem_0a...,3,1.153482,0.727766,0.474169,2.108951,0.502995,3,1.153482,2,...,0.502995,0.000,0.000000,0.000000,0.000000,0.00,0.497005,NaN,NaN,NaN
3,fr.inra.agrosyst.api.entities.GrowingSystem_0f...,10,2.568579,0.773219,0.230028,4.347294,0.375000,10,2.568579,5,...,0.428750,0.125,0.125000,0.375000,0.071250,0.04,0.050000,0.025000,0.528750,0.025000
4,fr.inra.agrosyst.api.entities.GrowingSystem_0f...,6,2.584963,1.000000,0.166667,6.000000,0.166667,6,2.584963,4,...,0.333333,0.000,0.000000,0.000000,0.333340,0.00,0.166667,0.166667,0.166667,0.166667
5,fr.inra.agrosyst.api.entities.GrowingSystem_11...,5,1.567635,0.675144,0.425890,2.348025,0.583836,7,2.351282,4,...,0.316043,0.000,0.072376,0.000000,0.000000,0.00,0.528347,NaN,0.036188,NaN
6,fr.inra.agrosyst.api.entities.GrowingSystem_18...,1,0.000000,NaN,1.000000,1.000000,1.000000,1,0.000000,1,...,1.000000,0.000,0.000000,0.000000,0.000000,0.00,NaN,NaN,NaN,NaN
7,fr.inra.agrosyst.api.entities.GrowingSystem_1b...,2,0.981917,0.981917,0.512482,1.951288,0.579000,2,0.981917,1,...,NaN,0.000,0.000000,0.000000,0.000000,0.00,NaN,NaN,NaN,NaN
8,fr.inra.agrosyst.api.entities.GrowingSystem_34...,5,2.010564,0.865903,0.280741,3.562003,0.409200,7,2.180109,3,...,0.232824,0.000,0.000000,0.142628,0.266573,0.00,NaN,0.215348,0.106971,0.215348
9,fr.inra.agrosyst.api.entities.GrowingSystem_3c...,3,1.249845,0.788564,0.450660,2.218966,0.492686,3,1.249845,1,...,0.547299,0.000,0.000000,0.000000,0.000000,0.00,NaN,NaN,NaN,NaN


In [13]:
def creer_df_tests(df, test_id, nb_par_colonne):
    lignes = []

    for colonne, n in nb_par_colonne.items():

        if colonne not in df.columns:
            raise ValueError(f"La colonne '{colonne}' n'existe pas.")

        serie = df[colonne]

        if n > len(serie):
            raise ValueError(
                f"Impossible de tirer {n} valeurs sans remise dans la colonne '{colonne}' "
                f"(seulement {len(serie)} valeurs disponibles)."
            )

        echantillon = serie.sample(n=n, replace=False)

        for idx, valeur in echantillon.items():
            lignes.append({
                "test_id": test_id,
                "index": idx,
                "valeur": valeur,
                "resultat": None,   # colonne vide
                "colonne": colonne
            })

    return pd.DataFrame(lignes)

nb_par_colonne = {
            'typo_espece_richesse':20, 
            'typo_espece_shannon':20,
            'typo_espece_evenness':15, 
            'typo_espece_simpson':15,
            'typo_espece_inverse_simpson':15, 
            'typo_espece_proportion_max':15,

            'espece_bota_richesse':10,
            'espece_bota_shannon':10,

            'famille_bota_richesse':10,
            'famille_bota_shannon':10,
            'famille_bota_Fabaceae':10,
            'famille_bota_Poaceae':10,
            'famille_bota_Brassicaceae':10, 
            'famille_bota_Autres':10,

            'saison_semis_richesse':10,
            'saison_semis_shannon':10,
            'saison_semis_printemps':10,
            'saison_semis_ete':10,
            'saison_semis_automne':10,
            'saison_semis_hiver':10,

            'prop_culture_avec_compagne':30,
            'prop_association':30,
            'prop_prairie':30,
            'prop_culture_intermédiaire':30,
            'prop_culture_porte_graine':50,
        }

# result.set_index('sdc_id', inplace=True)

final_TU = creer_df_tests(result,
                          'test_get_indicateur_diversite_outils_dirodur', 
                          nb_par_colonne)

final_TU.to_csv('/home/tbadie/Bureau/TU_a_utiliser.csv', index=False)